# Problématique

# Analyse Exploratoire des Données

#### Vérification GPU + imports

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
import shutil 

# Vérifier le GPU
print(f'GPU Disponible :{tf.config.list_physical_devices('GPU')}')
print(f'Version disponible : {tf.__version__}')

2026-04-22 15:52:43.093731: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776873163.261889      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776873163.312804      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776873163.715731      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776873163.715780      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776873163.715783      55 computation_placer.cc:177] computation placer alr

#### Vérification du dataset

In [ ]:

base_dir = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

print(os.listdir(base_dir))
for val in os.listdir(base_dir):
    pd.value_counts(val)


#### Bacteria vs Virus

In [ ]:
pneumonia = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train/PNEUMONIA/'

bacteria = [f for f in os.listdir(pneumonia) if 'bacteria' in f]

virus = [f for f in os.listdir(pneumonia) if 'virus' in f]
print(len(bacteria))
print(len(virus))



#### Distribution des 3 classes

In [ ]:
normal_n    = len(os.listdir(os.path.join(base_dir, "train", "NORMAL")))
bacteria_n  = len(bacteria)
virus_n     = len(virus)

classes = ["Normal", "Bactérienne", "Virale"]
counts  = [normal_n, bacteria_n, virus_n]
colors  = ["#5DCAA5", "#7F77DD", "#F0997B"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(classes, counts, color=colors, width=0.5)
ax.set_title("Distribution des classes - Train set", fontsize=13, pad=12)
ax.set_ylabel("Nombre d'images")

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(count), ha='center', fontsize=11, fontweight='bold')

sns.despine()
plt.tight_layout()
plt.savefig("distribution_classes.png", dpi=150) 
plt.show()

#### Visualiser des images des 3 classes côte à côte

In [ ]:
train_dir = os.path.join(base_dir, "train")
normal = os.path.join(train_dir, "NORMAL")
normal_files = os.listdir(normal)


def afficher_images(dossier, fichiers, titre, n=5):
    fig, ax = plt.subplots(1, n, figsize=(14, 6))
    
    for i in range(n):
        chemin = os.path.join(dossier, fichiers[i])  # chemin complet
        img = Image.open(chemin).convert("RGB")      # ouvrir image
        
        ax[i].imshow(img)                            # afficher
        ax[i].axis('off')                            # enlever axes
    
    fig.suptitle(titre)
    plt.tight_layout()
    plt.show()

afficher_images(pneumonia, bacteria, "Images Bactériennes")
afficher_images(pneumonia, virus, "Images Virales")
afficher_images(normal, normal_files, "Images Normales")

#### Vérifier la variabilité des tailles d'images

In [ ]:
widths, heights = [], []

# Échantillon d'images
all_files = (
    [(normal, f) for f in normal_files[:34]] +
    [(pneumonia, f) for f in bacteria[:33]] +
    [(pneumonia, f) for f in virus[:33]]
)

for folder, fname in all_files:
    try:
        chemin = os.path.join(folder, fname)
        w, h = Image.open(chemin).size
        
        widths.append(w)
        heights.append(h)
    except:
        pass

print(f"Largeur  - min: {min(widths)}  max: {max(widths)}  moyenne: {int(np.mean(widths))}")
print(f"Hauteur  - min: {min(heights)} max: {max(heights)} moyenne: {int(np.mean(heights))}")

# Pipeline de données

#### Réorganiser les fichiers

In [ ]:
import os
import shutil

base = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray'
dest = '/kaggle/working/chest_xray'

# Créer tous les dossiers
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'BACTERIA', 'VIRUS']:
        os.makedirs(f"{dest}/{split}/{cls}", exist_ok=True)

# Copier NORMAL
for split in ['train', 'val', 'test']:
    source = f"{base}/{split}/NORMAL"
    destination = f"{dest}/{split}/NORMAL"
    for fichier in os.listdir(source):
        shutil.copy(os.path.join(source, fichier), destination)

# Trier PNEUMONIA en BACTERIA et VIRUS
for split in ['train', 'val', 'test']:
    source = f"{base}/{split}/PNEUMONIA"
    for fichier in os.listdir(source):
        if 'bacteria' in fichier.lower():
            shutil.copy(os.path.join(source, fichier), f"{dest}/{split}/BACTERIA")
        elif 'virus' in fichier.lower():
            shutil.copy(os.path.join(source, fichier), f"{dest}/{split}/VIRUS")

# Vérifier la structure
print("VÉRIFICATION DE LA STRUCTURE\n")
for split in ['train', 'val', 'test']:
    print(f"{split.upper()}/")
    for cls in ['NORMAL', 'BACTERIA', 'VIRUS']:
        n = len(os.listdir(f"{dest}/{split}/{cls}"))
        print(f"  {cls} : {n} images")
    print()

#### Créer les datasets avec image_dataset_from_directory (3 classes)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/chest_xray/train',
    image_size=(224, 224),
    batch_size=32,
    shuffle=True,
    seed=42
)



In [ ]:
print(train_ds.class_names)
# ['BACTERIA', 'NORMAL', 'VIRUS']

# Voir la forme d'un batch
for images, labels in train_ds.take(1):
    print(images.shape)  # (32, 224, 224, 3)
    print(labels.shape)  # (32,)
    print(labels)        # [0, 1, 2, 0, 1, ...]

In [ ]:
val_ds= tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/chest_xray/val',
    image_size=(224, 224),
    batch_size=32,
    shuffle=False,# car on a pas besoin  de modifier l'ordre popur val et test 
    seed=42
)

In [ ]:
print(val_ds.class_names)

for images , labels in val_ds.take(1):
    print(images.shape)
    print(labels.shape)
    print(labels)

In [ ]:
test_ds= tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/chest_xray/test',
    image_size=(224, 224),
    batch_size=32,
    shuffle=False,# car on a pas besoin  de modifier l'ordre popur val et test 
    seed=42
)


#### Normalisation

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds   = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))

#### Déséquilibre des classes

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Récupérer tous les labels du train
labels = []
for images, label_batch in train_ds:
    labels.extend(label_batch.numpy())

labels = np.array(labels)

# Calculer les poids
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

#### Optimisation du pipeline 

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)